# Параметрическое исследование модели Daisyworld

В данном скрипте выполняется систематическое исследование влияния
различных параметров на поведение модели Daisyworld. Для каждого набора
параметров генерируются изображения состояния мира на разных этапах
эволюции (шаги 0, 5 и 40).

## Инициализация проекта и загрузка пакетов

In [ ]:
using DrWatson
@quickactivate "project"

using Agents
using DataFrames
using Plots
using CairoMakie

### Подключение модели

Импортируем определение модели Daisyworld из исходного файла.

In [ ]:
include(srcdir("daisyworld.jl"))

## Определение параметров эксперимента

### Структура параметров

Для исследования задаётся словарь параметров, где некоторые параметры
представлены в виде векторов. Это позволяет автоматически генерировать
все возможные комбинации значений.

**Исследуемые параметры:**
- `max_age` — максимальный возраст маргариток (25 и 40)
- `init_white` — начальная доля белых маргариток (0.2 и 0.8)

**Фиксированные параметры:**
- `griddims` — размер сетки (30×30)
- `init_black` — начальная доля чёрных маргариток (0.2)
- `albedo_white` — альбедо белых маргариток (0.75)
- `albedo_black` — альбедо чёрных маргариток (0.25)
- `surface_albedo` — альбедо почвы (0.4)
- `solar_change` — скорость изменения светимости (0.005)
- `solar_luminosity` — начальная светимость (1.0)
- `scenario` — сценарий изменения светимости (:default)
- `seed` — начальное значение для генератора случайных чисел (165)

In [ ]:
param_dict = Dict(
    :griddims => (30, 30),
    :max_age => [25, 40],
    :init_white => [0.2, 0.8],
    :init_black => 0.2,
    :albedo_white => 0.75,
    :albedo_black => 0.25,
    :surface_albedo => 0.4,
    :solar_change => 0.005,
    :solar_luminosity => 1.0,
    :scenario => :default,
    :seed => 165,
)

## Генерация комбинаций параметров

Функция `dict_list` из пакета DrWatson создаёт все возможные комбинации
параметров из заданного словаря. Для каждого параметра, представленного
вектором, генерируются отдельные эксперименты.

In [ ]:
params_list = dict_list(param_dict)

## Цикл по всем комбинациям параметров

Для каждого набора параметров выполняется:
1. Создание модели с заданными параметрами
2. Визуализация начального состояния
3. Выполнение 5 шагов и визуализация
4. Выполнение ещё 40 шагов и визуализация
5. Сохранение всех изображений с уникальными именами

In [ ]:
for params in params_list

### Создание модели

Модель инициализируется с текущим набором параметров.
Используется синтаксис `;params...` для распаковки словаря
в именованные аргументы.

In [ ]:
    model = daisyworld(; params...)

### Определение цвета агента

Функция `daisycolor` возвращает цвет маргаритки в зависимости от её вида.

In [ ]:
    daisycolor(a::Daisy) = a.breed

### Параметры визуализации

Задаём общие параметры отображения для всех графиков:
- `agent_color` — функция определения цвета агента
- `agent_size` — размер маргаритки
- `agent_marker` — символ для отображения маргаритки (цветок ✿)
- `heatarray` — массив для тепловой карты (температура)
- `heatkwargs` — параметры тепловой карты (диапазон от -20 до 60°C)

In [ ]:
    plotkwargs = (
        agent_color = daisycolor,
        agent_size = 20,
        agent_marker = '✿',
        heatarray = :temperature,
        heatkwargs = (colorrange = (-20, 60),),
    )

### Визуализация начального состояния (шаг 0)

Отображаем мир в начальный момент времени.

In [ ]:
    plt1, _ = abmplot(model; plotkwargs...)

### Визуализация после 5 шагов

Выполняем 5 шагов модели и отображаем результат.

In [ ]:
    step!(model, 5)
    plt2, _ = abmplot(model; heatarray = model.temperature, plotkwargs...)

### Визуализация после 40 шагов

Выполняем ещё 40 шагов (всего 45 шагов от начала) и отображаем результат.

In [ ]:
    step!(model, 40)
    plt3, _ = abmplot(model; heatarray = model.temperature, plotkwargs...)

### Формирование имён файлов

Используем функцию `savename` из пакета DrWatson для автоматического
формирования уникальных имён файлов на основе параметров эксперимента.
Это обеспечивает воспроизводимость и удобство идентификации результатов.

In [ ]:
    plt1_name = savename("daisyworld", params) * "_step01" * ".png"
    plt2_name = savename("daisyworld", params) * "_step04" * ".png"
    plt3_name = savename("daisyworld", params) * "_step40" * ".png"

### Сохранение результатов

Все три изображения сохраняются в каталог `plots/` с соответствующими
именами.

In [ ]:
    save(plotsdir(plt1_name), plt1)
    save(plotsdir(plt2_name), plt2)
    save(plotsdir(plt3_name), plt3)
end

## Интерпретация результатов

После выполнения скрипта в каталоге `plots/` появятся изображения для
каждой комбинации параметров. Анализ этих изображений позволяет:

1. **Исследовать влияние максимального возраста маргариток**:
   - При `max_age = 25` популяция обновляется быстрее
   - При `max_age = 40` популяция более стабильна

2. **Исследовать влияние начального соотношения видов**:
   - При `init_white = 0.2` доминируют чёрные маргаритки
   - При `init_white = 0.8` доминируют белые маргаритки

3. **Наблюдать за динамикой формирования кластеров**:
   - На шаге 0 — случайное распределение
   - На шаге 5 — начинается локальное изменение температуры
   - На шаге 40 — формируются устойчивые кластеры

4. **Оценивать способность системы к саморегуляции**:
   - Несмотря на разные начальные условия, система стремится
     к равновесному состоянию, соответствующему параметрам среды